# Exploratory Data Analysis

In [38]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
from datetime import UTC, datetime

import pandas as pd
import plotly.express as px
from tqdm import tqdm

from solar_predictor.storage import find_files_at_loc, get_location_prefix, read_csv

months = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

# Read Data

In [40]:
loc_lat, loc_lon = 42.0, 21.43

csvs = find_files_at_loc(get_location_prefix(loc_lat, loc_lon), "/sarah3.csv")
dfs = [read_csv(csv) for csv in tqdm(csvs, desc="Reading CSVs")]
raw_df = pd.concat(dfs, ignore_index=True)

Reading CSVs: 100%|██████████| 37/37 [00:14<00:00,  2.60it/s]


In [41]:
df = raw_df.copy()

df["datetime"] = pd.to_datetime(df["datetime"])

df["date"] = df["datetime"].dt.strftime("%Y-%m-%d")
df["month"] = df["datetime"].dt.month
df["month_day"] = df["datetime"].dt.strftime("%m-%d")
df["year"] = df["datetime"].dt.year

df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,datetime,date,month,month_day,year
0,631152000,0.0,0.0,0.0,0.0,1990-01-01 00:00:00+00:00,1990-01-01,1,01-01,1990
1,631153800,0.0,0.0,0.0,0.0,1990-01-01 00:30:00+00:00,1990-01-01,1,01-01,1990
2,631155600,0.0,0.0,0.0,0.0,1990-01-01 01:00:00+00:00,1990-01-01,1,01-01,1990
3,631157400,0.0,0.0,0.0,0.0,1990-01-01 01:30:00+00:00,1990-01-01,1,01-01,1990
4,631159200,0.0,0.0,0.0,0.0,1990-01-01 02:00:00+00:00,1990-01-01,1,01-01,1990


# Data Quality

## Missing and Duplicated Timestamps

In [42]:
expected = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="30min")
missing = expected.difference(df["datetime"])

print(f"Expected: {len(expected)}, actual: {len(df)}, missing: {len(missing)}")
print(f"Duplicated timestamps: {df['datetime'].duplicated().sum()}")

Expected: 645648, actual: 645648, missing: 0
Duplicated timestamps: 0


There are no missing or duplicated timestamps.

## Missing Values (in the rows)

### Missing Values in the tail
The satellite needs time to send the data to planet Earth, remove the last rows since they don't contain data.

In [43]:
last_complete_row = df[(~df.isna()).all(axis=1)].iloc[-1]

df = df.loc[lambda x: x["timestamp"] <= last_complete_row["timestamp"]]
df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,datetime,date,month,month_day,year
645547,1793136600,0.0,0.0,0.0,0.0,2026-10-27 21:30:00+00:00,2026-10-27,10,10-27,2026
645548,1793138400,0.0,0.0,0.0,0.0,2026-10-27 22:00:00+00:00,2026-10-27,10,10-27,2026
645549,1793140200,0.0,0.0,0.0,0.0,2026-10-27 22:30:00+00:00,2026-10-27,10,10-27,2026
645550,1793142000,0.0,0.0,0.0,0.0,2026-10-27 23:00:00+00:00,2026-10-27,10,10-27,2026
645551,1793143800,0.0,0.0,0.0,0.0,2026-10-27 23:30:00+00:00,2026-10-27,10,10-27,2026


In [44]:
df.count()

timestamp                   645552
shortwave_radiation         641974
direct_radiation            641962
diffuse_radiation           641962
direct_normal_irradiance    641962
datetime                    645552
date                        645552
month                       645552
month_day                   645552
year                        645552
dtype: int64

Although there are no missing timestamps, there are missing values withing the valid timestamps.

In [45]:
missing_count = df.groupby("year").apply(lambda x: x.isna().sum()).reset_index()
px.line(
    missing_count,
    x="year",
    y="shortwave_radiation",
    title="Number of missing periods per year",
)

Almost all years have some missing values.

### Missing Ranges

In [46]:
missing_dt = df[df.isna().any(axis=1)]["datetime"].reset_index(drop=True)
new_range = (missing_dt.diff() != pd.Timedelta("30min")).cumsum()
ranges = missing_dt.groupby(new_range).agg(start="first", end="last", steps="size")

ranges["hours"] = ranges["steps"] / 2
ranges["start"] = ranges["start"].dt.strftime("%Y-%m-%d %H:%M")
ranges["end"] = ranges["end"].dt.strftime("%Y-%m-%d %H:%M")
ranges = ranges.reset_index(drop=True)
ranges.to_csv("missing_ranges.csv", index=False)
ranges.head(10)

,start,end,steps,hours
0,1990-01-02 10:00,1990-01-02 18:00,17,8.5
1,1990-01-27 18:00,1990-01-28 06:00,25,12.5
2,1990-01-28 17:00,1990-01-28 17:00,1,0.5
3,1990-01-28 18:00,1990-01-29 06:00,25,12.5
4,1990-01-29 17:00,1990-01-29 17:00,1,0.5
5,1990-01-29 18:00,1990-01-30 03:30,20,10.0
6,1990-02-01 00:00,1990-02-01 00:00,1,0.5
7,1990-02-05 18:00,1990-02-06 06:00,25,12.5
8,1990-02-06 17:00,1990-02-06 17:00,1,0.5
9,1990-02-06 18:00,1990-02-06 20:30,6,3.0


#### Duration of Missing Ranges

In [47]:
pd.Series({h: (ranges["hours"] <= h).mean() for h in [1, 2, 4, 8, 24, 100]}).round(2)

1      0.71
2      0.92
4      0.94
8      0.95
24     0.99
100    1.00
dtype: float64

Most of the missing periods (90%) are up to 2 hours long,
however there are periods (5%) with more than 8 hours of missing data.
These were probably because of bigger satellite outages.

#### Missing Range Sample

In [ ]:
missing_sample = df.loc[
    lambda x: (x["date"] >= "1990-01-25") & (x["date"] <= "1990-02-01")
]
px.line(missing_sample, x="datetime", y="shortwave_radiation")

In [ ]:
missing_sample = df.loc[
    lambda x: (x["date"] >= "2005-04-16") & (x["datetime"] <= "2005-04-20")
]
px.line(missing_sample, x="datetime", y="shortwave_radiation")

# Charts

In [50]:
daily = df.groupby("date").agg({"shortwave_radiation": "mean"}).reset_index()

fig = px.line(
    daily,
    x="date",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "date": "Date"},
    title="Daily Shortwave Radiation Over Time",
)
end = daily["date"].max()
fig.update_layout(
    xaxis={
        "rangeslider_visible": True,
        "range": [end - pd.DateOffset(years=5), end],
    },
)
fig

TypeError: unsupported operand type(s) for -: 'str' and 'DateOffset'

# Shortwave Radiation by Months

In [ ]:
daily_month = (
    df.groupby(["month", "date"]).agg({"shortwave_radiation": "mean"}).reset_index()
)


fig = px.box(
    daily_month,
    x="month",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "month": "Month"},
    title="Daily Mean Shortwave Radiation by Month",
)
fig.update_xaxes(
    type="category",
    tickvals=list(range(1, 13)),
    ticktext=months,
)
fig